# Save Tuned Models — Home Credit

Train hyperparameter-tuned regression models to predict **AMT_CREDIT** (loan amount), evaluate on a held-out test set, save all fitted models to `models/`, and persist the best model by test **R²**.

## 1. Imports

In [ ]:
import json
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

RANDOM_STATE = 42
CV_FOLDS = 3
PROJECT_ROOT = Path(r"c:\Users\Admin\Desktop\Home Credit")
MODELS_DIR = PROJECT_ROOT / "models"

## 2. Load data & split

Same pipeline as `model_comparison.ipynb` and `hyperparameter_tuning.ipynb`: load `application_train.csv`, select target and features, `dropna()`, 80/20 split with `random_state=42`.

In [ ]:
DATA_PATH = PROJECT_ROOT / "csv" / "application_train.csv"

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")

target = "AMT_CREDIT"
feature_cols = [
    "AMT_INCOME_TOTAL",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "CNT_CHILDREN",
    "DAYS_BIRTH",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
]

model_df = df[[target] + feature_cols].copy()
rows_before = len(model_df)
model_df = model_df.dropna()
rows_after = len(model_df)

print(f"Rows before dropna: {rows_before:,}")
print(f"Rows after dropna:  {rows_after:,}")
print(f"Dropped rows:       {rows_before - rows_after:,}")

X = model_df[feature_cols]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Train size: {len(X_train):,} samples")
print(f"Test size:  {len(X_test):,} samples")

## 3. Train + tune each model, evaluate on test set

Search spaces match `hyperparameter_tuning.ipynb`. Ridge, Lasso, and ElasticNet use a `StandardScaler` pipeline. Random Forest uses `RandomizedSearchCV` for speed.

In [ ]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    return r2, mae, rmse


def run_search(name, search, X_train, y_train):
    print(f"Tuning {name}...")
    search.fit(X_train, y_train)
    print(f"  Best params: {search.best_params_}")
    print(f"  Best CV score (neg MSE): {search.best_score_:.2f}")
    print()
    return search.best_estimator_, search.best_params_


tuned_models = {}
best_params = {}

# Linear Regression — untuned baseline
lr = LinearRegression()
lr.fit(X_train, y_train)
tuned_models["Linear Regression"] = lr
best_params["Linear Regression"] = "default (no tuning)"
print("Linear Regression: fitted with default params (baseline)\n")

# Ridge
ridge_search = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("model", Ridge())]),
    param_grid={"model__alpha": [0.01, 0.1, 1, 10, 100]},
    cv=CV_FOLDS,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)
tuned_models["Ridge"], best_params["Ridge"] = run_search(
    "Ridge", ridge_search, X_train, y_train
)

# Lasso
lasso_search = GridSearchCV(
    Pipeline([("scaler", StandardScaler()), ("model", Lasso(max_iter=10000))]),
    param_grid={"model__alpha": [0.001, 0.01, 0.1, 1, 10]},
    cv=CV_FOLDS,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)
tuned_models["Lasso"], best_params["Lasso"] = run_search(
    "Lasso", lasso_search, X_train, y_train
)

# ElasticNet
en_search = GridSearchCV(
    Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(max_iter=10000)),
    ]),
    param_grid={
        "model__alpha": [0.001, 0.01, 0.1, 1, 10],
        "model__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
    },
    cv=CV_FOLDS,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)
tuned_models["Elastic Net"], best_params["Elastic Net"] = run_search(
    "Elastic Net", en_search, X_train, y_train
)

# Decision Tree
dt_search = GridSearchCV(
    DecisionTreeRegressor(random_state=RANDOM_STATE),
    param_grid={
        "max_depth": [5, 10, 15, 20, None],
        "min_samples_split": [2, 5, 10, 20],
        "min_samples_leaf": [1, 2, 5, 10],
    },
    cv=CV_FOLDS,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)
tuned_models["Decision Tree"], best_params["Decision Tree"] = run_search(
    "Decision Tree", dt_search, X_train, y_train
)

# Random Forest — RandomizedSearchCV for speed
rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions={
        "n_estimators": [50, 100, 150, 200, 300],
        "max_depth": [5, 10, 15, 20, 25, None],
        "min_samples_split": [2, 5, 10, 20],
    },
    n_iter=20,
    cv=CV_FOLDS,
    scoring="neg_mean_squared_error",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
tuned_models["Random Forest"], best_params["Random Forest"] = run_search(
    "Random Forest", rf_search, X_train, y_train
)

# Evaluate on test set
results = []
for name, model in tuned_models.items():
    r2, mae, rmse = evaluate_model(model, X_test, y_test)
    results.append({
        "Model": name,
        "Best Params": best_params[name],
        "R²": r2,
        "MAE": mae,
        "RMSE": rmse,
    })
    print(f"{name} (test set):")
    print(f"  Best params: {best_params[name]}")
    print(f"  R²:   {r2:.4f}")
    print(f"  MAE:  {mae:,.2f}")
    print(f"  RMSE: {rmse:,.2f}")
    print()

## 4. Comparison table sorted by R²

In [ ]:
results_df = pd.DataFrame(results).sort_values(
    ["R²", "RMSE"], ascending=[False, True]
).reset_index(drop=True)
results_df

## 5. Save all models to `models/` folder

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

model_files = {
    "Linear Regression": "linear_regression.joblib",
    "Ridge": "ridge.joblib",
    "Lasso": "lasso.joblib",
    "Elastic Net": "elastic_net.joblib",
    "Decision Tree": "decision_tree.joblib",
    "Random Forest": "random_forest.joblib",
}

saved_paths = []
for name, filename in model_files.items():
    path = MODELS_DIR / filename
    joblib.dump(tuned_models[name], path)
    saved_paths.append(str(path))
    print(f"Saved {name} -> {path}")

metadata = []
for _, row in results_df.iterrows():
    params = row["Best Params"]
    if isinstance(params, dict):
        params_out = params
    else:
        params_out = str(params)
    metadata.append({
        "model_name": row["Model"],
        "filename": model_files[row["Model"]],
        "best_hyperparameters": params_out,
        "r2": float(row["R²"]),
        "mae": float(row["MAE"]),
        "rmse": float(row["RMSE"]),
    })

metadata_path = MODELS_DIR / "model_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)
saved_paths.append(str(metadata_path))
print(f"\nSaved metadata -> {metadata_path}")

## 6. Select and save best model

Best model = highest test **R²**; ties broken by lowest **RMSE**.

In [ ]:
best_row = results_df.iloc[0]
best_name = best_row["Model"]
best_model = tuned_models[best_name]

best_model_path = MODELS_DIR / "best_model.joblib"
best_model_root_path = PROJECT_ROOT / "best_model.joblib"

joblib.dump(best_model, best_model_path)
joblib.dump(best_model, best_model_root_path)
saved_paths.extend([str(best_model_path), str(best_model_root_path)])

print(f"Best model: {best_name}")
print(f"  Best params: {best_row['Best Params']}")
print(f"  R²:   {best_row['R²']:.4f}")
print(f"  MAE:  {best_row['MAE']:,.2f}")
print(f"  RMSE: {best_row['RMSE']:,.2f}")
print(f"\nSaved best model -> {best_model_path}")
print(f"Saved best model -> {best_model_root_path}")

## 7. Verify loads work

Load `best_model.joblib` and predict on a few test rows.

In [ ]:
loaded_model = joblib.load(best_model_path)
sample_X = X_test.head(5)
sample_y = y_test.head(5)
sample_pred = loaded_model.predict(sample_X)

verify_df = sample_X.copy()
verify_df["actual_AMT_CREDIT"] = sample_y.values
verify_df["predicted_AMT_CREDIT"] = sample_pred
verify_df["error"] = verify_df["actual_AMT_CREDIT"] - verify_df["predicted_AMT_CREDIT"]

print(f"Loaded model type: {type(loaded_model).__name__}")
print("Sample predictions vs actual:")
display(verify_df)

print("\nAll saved files:")
for p in saved_paths:
    exists = os.path.exists(p)
    size_kb = os.path.getsize(p) / 1024 if exists else 0
    print(f"  {'OK' if exists else 'MISSING'}  {p}  ({size_kb:.1f} KB)")